In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from decimal import Decimal

dbutils.widgets.text("catalogo", "catalog_dev")
catalogo = dbutils.widgets.get("catalogo").strip()
if catalogo not in ("catalog_dev", "catalog_prod"):
    raise ValueError("Catálogo no permitido: usar catalog_dev o catalog_prod")

fp = spark.table(f"{catalogo}.golden.fact_pedidos")
fv = spark.table(f"{catalogo}.golden.fact_ventas")
fg = spark.table(f"{catalogo}.golden.fact_pagos")
fr = spark.table(f"{catalogo}.golden.fact_resenas")

print("Catálogo:", catalogo)
print("Tablas de hechos cargadas para validación")

In [0]:
lineas = fv.agg(
    F.count("*").alias("total_lineas"),
    F.sum("precio").alias("importe_articulos"),
    F.sum("valor_flete").alias("importe_flete")
).first()

pedidos_detalle = fp.filter(F.col("tiene_detalle") == 1)
res_pedidos = pedidos_detalle.agg(
    F.count("*").alias("pedidos_con_detalle"),
    F.sum("importe_articulos").alias("importe_articulos"),
    F.sum("importe_flete").alias("importe_flete")
).first()

sin_detalle = fp.filter(F.col("tiene_detalle") == 0)
if sin_detalle.filter(
    F.col("importe_articulos").isNotNull() | F.col("importe_flete").isNotNull()
).limit(1).count() != 0:
    raise ValueError("Hay importes atribuidos a pedidos sin detalle")

def dinero(valor):
    return Decimal(str(valor or 0)).quantize(Decimal("0.01"))

if dinero(lineas["importe_articulos"]) != dinero(res_pedidos["importe_articulos"]):
    raise ValueError("Diferencia de importes de artículos entre fact_ventas y fact_pedidos")
if dinero(lineas["importe_flete"]) != dinero(res_pedidos["importe_flete"]):
    raise ValueError("Diferencia de importes de flete entre fact_ventas y fact_pedidos")

print(f"Líneas de ventas: {lineas['total_lineas']:,}")
print(f"Pedidos con detalle: {res_pedidos['pedidos_con_detalle']:,}")
print(f"Pedidos sin detalle: {sin_detalle.count():,}")
print(f"Importe artículos conciliado: R$ {dinero(lineas['importe_articulos']):,.2f}")
print(f"Importe flete conciliado: R$ {dinero(lineas['importe_flete']):,.2f}")
print("OK: importes de ventas y pedidos coinciden; no hay duplicación")

In [0]:

k = pedidos_detalle.agg(
    F.avg("importe_articulos").alias("ticket_sin_flete"),
    F.avg("importe_total").alias("ticket_con_flete"),
    F.avg("cantidad_articulos").alias("articulos_por_pedido")
).first()


frecuencia_clientes = (
    fp.filter(F.col("id_cliente_unico").isNotNull())
    .groupBy("id_cliente_unico")
    .agg(F.count("*").alias("n_pedidos"))
)
r = frecuencia_clientes.agg(
    F.count("*").alias("clientes"),
    F.sum(F.when(F.col("n_pedidos") >= 2, 1).otherwise(0)).alias("recurrentes")
).first()
porcentaje_recurrentes = 100.0 * r["recurrentes"] / r["clientes"] if r["clientes"] else 0.0


entregas_evaluables = fp.filter(
    (F.col("estado_pedido") == "delivered") &
    F.col("entrega_a_tiempo").isin(0, 1)
)
e = entregas_evaluables.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("entrega_a_tiempo") == 1, 1).otherwise(0)).alias("puntuales"),
    F.avg("dias_entrega").alias("dias_promedio")
).first()
porcentaje_puntuales = 100.0 * e["puntuales"] / e["total"] if e["total"] else 0.0


s = fr.agg(
    F.avg("calificacion").alias("promedio"),
    F.count("*").alias("resenas"),
    F.sum(F.when(F.col("calificacion") <= 2, 1).otherwise(0)).alias("bajas")
).first()

print("KPIs DE OLIST (REALES BRASILEÑOS)")
print(f"Ticket promedio (sin flete): R$ {float(k['ticket_sin_flete']):,.2f}")
print(f"Ticket promedio (con flete): R$ {float(k['ticket_con_flete']):,.2f}")
print(f"Artículos por pedido con detalle: {float(k['articulos_por_pedido']):.2f}")
print(f"Clientes recurrentes: {r['recurrentes']:,} / {r['clientes']:,} = {porcentaje_recurrentes:.2f}%")
print(f"Entregas a tiempo: {e['puntuales']:,} / {e['total']:,} = {porcentaje_puntuales:.2f}%")
print(f"Días promedio de entrega (muestra comparable): {float(e['dias_promedio'] or 0):.2f}")
print(f"Calificación promedio: {float(s['promedio']):.2f}/5")
print(f"Evaluaciones bajas (1 o 2): {s['bajas']:,} / {s['resenas']:,}")


if abs(float(k["ticket_sin_flete"]) - 137.75) > 0.02:
    raise ValueError("Ticket promedio difiere de la auditoría inicial; revisar cálculo")
print("OK: ticket promedio conciliado con el cálculo anterior")

In [0]:
referencias = [
    ("fact_ventas", "id_fecha", "dim_fecha", "id_fecha"),
    ("fact_pedidos", "id_fecha", "dim_fecha", "id_fecha"),
    ("fact_pagos", "id_fecha", "dim_fecha", "id_fecha"),
    ("fact_resenas", "id_fecha", "dim_fecha", "id_fecha"),
    ("fact_ventas", "id_categoria", "dim_categoria", "id_categoria"),
]

for hecho, col_hecho, dimension, col_dimension in referencias:
    claves_hecho = (
        spark.table(f"{catalogo}.golden.{hecho}")
        .select(F.col(col_hecho).alias("clave"))
        .filter(F.col("clave").isNotNull()).distinct()
    )
    claves_dimension = (
        spark.table(f"{catalogo}.golden.{dimension}")
        .select(F.col(col_dimension).alias("clave")).distinct()
    )
    sin_dim = claves_hecho.join(claves_dimension, "clave", "left_anti").count()
    print(f"{hecho}.{col_hecho} -> {dimension}: {sin_dim} claves sin dimensión")
    if sin_dim:
        raise ValueError(f"Integridad referencial fallida: {hecho} -> {dimension}")

print("OK: claves de fecha y categoría corresponden a dimensiones")

In [0]:
### Ticket medio:
ticket_mensual = (
    pedidos_detalle.filter(F.col("id_fecha").isNotNull())
    .withColumn("mes", F.substring(F.col("id_fecha").cast("string"), 1, 6))
    .groupBy("mes")
    .agg(
        F.count("*").alias("pedidos"),
        F.round(F.avg("importe_articulos"), 2).alias("ticket_promedio")
    )
    .orderBy("mes")
)

print("EVOLUCIÓN MENSUAL DEL TICKET")
display(ticket_mensual)


productos_comparables = (
    fv.filter((F.col("precio").isNotNull()) & (F.col("id_fecha").isNotNull()))
    .withColumn("mes", F.substring(F.col("id_fecha").cast("string"), 1, 6))
    .groupBy("id_producto")
    .agg(
        F.count("*").alias("articulos"),
        F.countDistinct("mes").alias("meses_observados"),
        F.countDistinct("precio").alias("precios_distintos")
    )
    .filter((F.col("meses_observados") >= 3) & (F.col("precios_distintos") >= 2))
    .orderBy(F.desc("articulos"))
)
print("PRODUCTOS CANDIDATOS PARA TENDENCIA DE PRECIOS (top 10)")
display(productos_comparables.limit(10))
print("VALIDACIÓN MODELO Y KPIs COMPLETADA")